# Alberta Wildfire Classifier — Data Preview

## Exploratory data analysis | Alberta wildfires (2006–2025)

**Objective:** Characterize historical wildfire records, assess data quality, and examine attributes relevant to predicting final `SIZE_CLASS` (A–E) at initial assessment.

**Scope:** Raw-data profiling and exploratory visualization. This analysis does not modify source data or perform model training.

**Source:** [Government of Alberta historical wildfire data](https://open.canada.ca/data/en/dataset/a221e7a0-4f46-4be7-9c5a-e29de9a3447e) · [Data dictionary, 2006–2025](https://open.alberta.ca/dataset/a221e7a0-4f46-4be7-9c5a-e29de9a3447e/resource/1b635b8b-a937-4be4-857e-8aeef77365d2/download/fp-historical-wildfire-data-dictionary-2006-2025.pdf)


## 1. Data loading

Source file: `data/raw/fp-historical-wildfire-data-2006-2025.csv`. Paths are resolved from the repository root or `notebooks/` directory.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

DATA_FILE = Path("data/raw/fp-historical-wildfire-data-2006-2025.csv")
project_roots = [Path.cwd(), Path.cwd().parent]

project_root = next((p for p in project_roots if (p / DATA_FILE).is_file()), None)
if project_root is None:
    raise FileNotFoundError(
        f"Could not find {DATA_FILE}. Open Jupyter from the repository root "
        "or notebooks/ directory, and confirm the raw CSV exists."
    )

data_path = project_root / DATA_FILE
df = pd.read_csv(data_path, low_memory=False)

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False})

print(f"Dataset: {data_path.relative_to(project_root)}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")

## 2. Dataset structure

Each record represents a historical wildfire incident. The raw schema includes information potentially recorded after initial assessment; predictive eligibility is assessed separately.


In [ ]:
display(df.head(8))

schema = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "missing": df.isna().sum().values,
})
display(schema)

In [ ]:
num_summary = df.select_dtypes(include="number").describe().T
if not num_summary.empty:
    display(num_summary.round(2))

## 3. Final wildfire size class distribution

`SIZE_CLASS` is the prediction target with categories A–E. Target frequency is used to assess class imbalance.


In [ ]:
class_order = ["A", "B", "C", "D", "E"]
y = df["SIZE_CLASS"].astype("string").str.strip().str.upper()
class_counts = y.value_counts().reindex(class_order, fill_value=0)
class_share = (class_counts / len(df) * 100).round(2)

display(pd.DataFrame({"records": class_counts, "share_%": class_share}))

fig, ax = plt.subplots()
bars = ax.bar(class_counts.index, class_counts.values, color="#2563eb")
for bar, pct in zip(bars, class_share):
    ax.annotate(f"{pct:.1f}%", (bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9)
ax.set(title="Final wildfire size class distribution", xlabel="Size class", ylabel="Number of records")
ax.margins(y=0.12)
plt.show()

**Evaluation implication:** A concentration of records in smaller size classes can make accuracy misleading. Macro F1-score, per-class recall, and confusion matrices provide complementary evaluation measures.


## 4. Wildfire record counts by year

This chart shows the number of records in the dataset for each reported year—not an estimate of total wildfire incidence outside the dataset's coverage.

In [ ]:
annual_counts = df["YEAR"].value_counts().sort_index()
fig, ax = plt.subplots()
ax.plot(annual_counts.index, annual_counts.values, color="#0f766e", marker="o", linewidth=1.8)
ax.set(title="Historical wildfire records by year", xlabel="Year", ylabel="Records")
ax.set_xticks(annual_counts.index[::max(1, len(annual_counts)//10)])
plt.show()

print(f"Year coverage in file: {df['YEAR'].min()}–{df['YEAR'].max()}")

## 5. Missing values and data types

Missingness is assessed on the **raw dataset**, including attributes that will eventually be excluded. Some suppression-stage fields may be absent because an event was not recorded; missingness alone does not prove an event never happened.

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str)
}).sort_values("missing_pct", ascending=False)

display(missing)

plot_missing = missing.loc[missing["missing_count"] > 0].head(15).sort_values("missing_pct")
if len(plot_missing):
    fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(plot_missing) + 1)))
    ax.barh(plot_missing.index, plot_missing["missing_pct"], color="#d97706")
    ax.set(title="Top 15 attributes with missing data", xlabel="Missing records (%)")
    ax.set_xlim(0, 105)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

## 6. Incident identifiers and duplicate records

The combination of `YEAR` and `FIRE_NUMBER` is examined for duplicate incident keys. Duplicate-key flags are reported without automatic removal.


In [ ]:
keys = ["YEAR", "FIRE_NUMBER"]
if all(col in df.columns for col in keys):
    duplicates = df.duplicated(subset=keys, keep=False)
    print(f"Records sharing a (YEAR, FIRE_NUMBER) key: {duplicates.sum():,}")
    if duplicates.any():
        display(df.loc[duplicates, keys + ["SIZE_CLASS"]].sort_values(keys).head(20))
else:
    print("Expected identifier columns are not available.")

print(f"Fully duplicated rows: {df.duplicated().sum():,}")

## 7. Numerical distributions and quality checks

The distributions of numerical assessment attributes provide a basis for identifying skewness, extreme values, and potentially invalid observations. The logarithmic transformation of assessment hectares is used for display only.


In [ ]:
variables = ["ASSESSMENT_HECTARES", "FIRE_SPREAD_RATE", "TEMPERATURE", "RELATIVE_HUMIDITY", "WIND_SPEED"]
display(df[variables].describe().T.round(2))

area = pd.to_numeric(df["ASSESSMENT_HECTARES"], errors="coerce").dropna()
area = area[area >= 0]
fig, ax = plt.subplots()
ax.hist(np.log1p(area), bins=45, color="#2563eb", edgecolor="white")
ax.set(title="Assessment size distribution (log1p scale)",
       xlabel="log(1 + assessment hectares)", ylabel="Number of wildfires")
plt.show()

In [ ]:
# Flag suspicious values for investigation; do not silently delete them.
quality_checks = {
    "Negative assessment hectares": (df["ASSESSMENT_HECTARES"] < 0).sum(),
    "Negative fire spread rate": (df["FIRE_SPREAD_RATE"] < 0).sum(),
    "Relative humidity outside 0–100": ((df["RELATIVE_HUMIDITY"] < 0) | (df["RELATIVE_HUMIDITY"] > 100)).sum(),
    "Negative wind speed": (df["WIND_SPEED"] < 0).sum(),
}
display(pd.Series(quality_checks, name="flagged_records").to_frame())

## 8. Forest-area information in `FIRE_NUMBER`

The first character of `FIRE_NUMBER` is examined as a potential forest-area prefix. Its geographic interpretation requires confirmation against the data dictionary.


In [ ]:
forest_area = df["FIRE_NUMBER"].astype("string").str[0]
preview = pd.DataFrame({"FIRE_NUMBER": df["FIRE_NUMBER"], "FOREST_AREA_PREFIX": forest_area})
display(preview.head(10))

forest_counts = forest_area.value_counts(dropna=False).sort_values(ascending=False)
display(forest_counts.rename("records").to_frame())
fig, ax = plt.subplots()
ax.bar(forest_counts.index.astype(str), forest_counts.values, color="#0f766e")
ax.set(title="Wildfire records by extracted forest-area prefix",
       xlabel="First character of FIRE_NUMBER", ylabel="Records")
plt.show()

## 9. Assessment size by final size class

Summary statistics describe the relationship between the size recorded at assessment and the eventual wildfire size class. These associations are descriptive rather than causal.


In [ ]:
assessment_by_class = (
    pd.DataFrame({"SIZE_CLASS": y, "ASSESSMENT_HECTARES": df["ASSESSMENT_HECTARES"]})
      .groupby("SIZE_CLASS")["ASSESSMENT_HECTARES"]
      .agg(records="count", median_hectares="median", mean_hectares="mean")
      .reindex(class_order)
)
display(assessment_by_class.round(3))

fig, ax = plt.subplots()
ax.bar(assessment_by_class.index, assessment_by_class["median_hectares"], color="#2563eb")
ax.set(title="Median size at assessment, grouped by final class",
       xlabel="Final size class", ylabel="Median assessment hectares")
plt.show()

## 10. Feature availability at prediction time

**Prediction point:** Initial assessment. Candidate features must reflect information available at or before `ASSESSMENT_DATETIME`. The table summarizes potential target leakage and feature-timing concerns.


In [ ]:
feature_audit = pd.DataFrame([
    ("SIZE_CLASS", "Target", "Outcome to predict; must not appear in input X"),
    ("CURRENT_SIZE", "Exclude", "Represents later size/outcome information"),
    ("FIRST_UC_SIZE", "Exclude", "Future suppression milestone"),
    ("FIRE_FIGHTING_START_SIZE", "Exclude", "Suppression-start measurement may occur after assessment"),
    ("FIRST_EX_DATE", "Exclude", "Future extinguishment milestone"),
    ("FIRE_NUMBER", "Transform", "Extract forest-area prefix; do not train on unique identifier"),
    ("ASSESSMENT_DATETIME", "Transform", "Consider month/season, not the raw timestamp"),
    ("ASSESSMENT_HECTARES", "Candidate", "Size measured at initial assessment"),
    ("FIRE_SPREAD_RATE", "Verify availability", "Check whether recorded at assessment or revised later"),
    ("TEMPERATURE", "Verify availability", "Check observation timing and data quality"),
    ("FUEL_TYPE", "Verify availability", "Check observation timing and missingness"),
    ("INITIAL_ACTION_BY", "Exclude unless verified", "Initial action may occur after assessment"),
], columns=["attribute", "proposed_treatment", "rationale"])
feature_audit["in_raw_file"] = feature_audit["attribute"].isin(df.columns)
display(feature_audit)

## 11. Data quality and modeling considerations

The exploratory analysis covers:

- **Class imbalance:** Distribution of final size classes A–E and implications for evaluation.
- **Missingness:** Availability of candidate assessment-time attributes.
- **Data validity:** Negative spread rates, out-of-range humidity, and other flagged values requiring source-code interpretation.
- **Geographic encoding:** Consistency of forest-area prefixes extracted from `FIRE_NUMBER`.
- **Feature timing:** Availability of weather, fuel, spread, and response attributes at initial assessment.
- **Evaluation design:** Reproducible data partitioning and potential later-year holdout evaluation.

**Classification methods:** Decision Tree, KNN, and Random Forest. Primary evaluation metrics include macro F1-score, per-class recall, and confusion matrices.

**Data provenance:** [Government of Alberta historical wildfire data](https://open.canada.ca/data/en/dataset/a221e7a0-4f46-4be7-9c5a-e29de9a3447e), [Open Government Licence – Alberta](https://open.alberta.ca/licence).
